In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib


df = pd.read_csv('../data/processed/nlp_processed_reviews.csv')
df.dropna(subset=['processed_text', 'sentiment'], inplace=True)


X = df['processed_text']
y = df['sentiment']

_, X_test_raw, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


best_rf = joblib.load('../models/sentiment_model.pkl')
tfidf = joblib.load('../models/tfidf_vectorizer.pkl')


X_test_tfidf = tfidf.transform(X_test_raw)


df_test = pd.DataFrame({'text': X_test_raw, 'actual': y_test, 'predicted': best_rf.predict(X_test_tfidf)})
errors = df_test[df_test['actual'].str.lower() != df_test['predicted']]

print("Modelin en çok hata yaptığı örnekler:")
print(errors.head(5))



Modelin en çok hata yaptığı örnekler:
                                                  text    actual predicted
187  great controller fire tv purchased 2 two probl...  Negative  positive
104  bought case time bought new fire tablet prime ...  Negative  positive
322  dont know whats wrong fire tv remote tend malf...  Negative  positive
254  seemed work fine tap open mic feature best bou...  Negative  positive
58   change battery remote twice per month since pu...  Negative  positive


"""
--- Önerilen Pipeline Taslağı: Haftalık Şikayet Özeti ---
1. Veri Toplama: Haftalık olarak yeni gelen tüm kullanıcı yorumları veritabanından çekilir.
2. Filtreleme (Bulanık Mantık): Sadece "Güvenilirlik Skoru" > 70 olan yorumlar işleme alınır.
3. Duygu Analizi: Eğitilen 'sentiment_model.pkl' ile yorumlar sınıflandırılır.
4. Kümeleme (Clustering): 'Negative' sınıfındaki yorumlar üzerinde LDA (Latent Dirichlet Allocation) 
   veya Keyword Extraction (anahtar kelime çıkarma) çalıştırılır.
5. Raporlama: En sık geçen negatif kelimeler (örn: 'şarj', 'ekran', 'donma') 
   Otomatik bir e-posta veya Slack bildirimi ile ilgili ürün ekibine raporlanır.
"""